In [ ]:
# Step 1: Clone ComfyUI
!git clone https://github.com/comfyanonymous/ComfyUI
%cd ComfyUI

# Step 2: Install dependencies
!pip install -q xformers==0.0.28.post1 --no-deps
!pip install -q -r requirements.txt

# Step 3: Create necessary directories
!mkdir -p models/checkpoints
!mkdir -p models/ipadapter
!mkdir -p models/clip_vision

# Step 4: Download RealVisXL model (6.5GB)
!wget -c https://civitai.com/api/download/models/361593 -O models/checkpoints/RealVisXL_V4.safetensors

# Step 5: Download IP-Adapter models
!wget -c https://huggingface.co/h94/IP-Adapter/resolve/main/sdxl_models/ip-adapter-plus-face_sdxl_vit-h.safetensors -O models/ipadapter/ip-adapter-plus-face_sdxl.safetensors

!wget -c https://huggingface.co/h94/IP-Adapter/resolve/main/models/image_encoder/model.safetensors -O models/clip_vision/CLIP-ViT-H-14.safetensors

# Step 6: Install ComfyUI Manager (for easier node management)
%cd custom_nodes
!git clone https://github.com/ltdrdata/ComfyUI-Manager.git
%cd ..

# Step 7: Install IP-Adapter nodes
%cd custom_nodes
!git clone https://github.com/cubiq/ComfyUI_IPAdapter_plus.git
%cd ..



Cloning into 'ComfyUI'...
remote: Enumerating objects: 28887, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 28887 (delta 3), reused 3 (delta 3), pack-reused 28881 (from 2)
Receiving objects: 100% (28887/28887), 75.41 MiB | 15.86 MiB/s, done.
Resolving deltas: 100% (19595/19595), done.
/content/ComfyUI
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 103.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.0/19.0 MB 124.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 149.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 131.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 110.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.3/39.3 MB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 5.7 MB/s eta 0:00:00


In [ ]:
%cd /content/ComfyUI

# Start ComfyUI in background
import subprocess
import time

# Start ComfyUI
comfy_process = subprocess.Popen(
    ['python', 'main.py', '--dont-print-server', '--listen', '0.0.0.0'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

print("⏳ Starting ComfyUI... (waiting 10 seconds)")
time.sleep(10)
print("✅ ComfyUI should be running now")

# Start Cloudflared tunnel
print("\n🌐 Creating Cloudflare tunnel...")
!./cloudflared-linux-amd64 tunnel --url http://localhost:8188


/content/ComfyUI
⏳ Starting ComfyUI... (waiting 10 seconds)
✅ ComfyUI should be running now

🌐 Creating Cloudflare tunnel...
2025-12-07T06:06:01Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2025-12-07T06:06:01Z INF Requesting new quick Tunnel on trycloudflare.com...
2025-12-07T06:06:06Z INF +--------------------------------------------------------------------------------------------+
2025-12-07T06:06:06Z INF |  Your quick Tunnel has been created! Visit it at 

In [ ]:
!pkill -9 -f "python.*main.py"
!pkill -9 -f "cloudflared"
!sleep 3
!echo "✓ All processes killed"

# Navigate to ComfyUI and clear logs
%cd /content/ComfyUI
!rm -f user/comfyui.log
!rm -rf user/__manager/*.log
!echo "✓ Log files deleted"

# Verify logs are gone
!ls -la user/ | grep -E "\.log|__manager"

# Create a backup and patch the logger
!cp app/logger.py app/logger.py.backup

# Patch the flush method to handle I/O errors gracefully
!cat > /tmp/logger_patch.py << 'EOF'
import sys

# Read the file
with open('/content/ComfyUI/app/logger.py', 'r') as f:
    content = f.read()

# Find and replace the flush method
old_flush = '''    def flush(self):
        super().flush()'''

new_flush = '''    def flush(self):
        try:
            super().flush()
        except (OSError, IOError):
            # Silently ignore I/O errors during flush (common in cloud environments)
            pass'''

if old_flush in content:
    content = content.replace(old_flush, new_flush)
    with open('/content/ComfyUI/app/logger.py', 'w') as f:
        f.write(content)
    print("✓ Logger patched successfully")
else:
    print("⚠ Logger already patched or structure changed")
'EOF'

!python3 /tmp/logger_patch.py



✓ All processes killed
/content/ComfyUI
✓ Log files deleted
-rw-r--r--  1 root root 36093 Dec  7 05:50 comfyui.prev2.log
-rw-r--r--  1 root root  6520 Dec  7 05:46 comfyui.prev.log
drwxr-xr-x  5 root root  4096 Dec  7 05:06 __manager
/bin/bash: line 1: warning: here-document at line 1 delimited by end-of-file (wanted `EOF')
⚠ Logger already patched or structure changed
